# Error analysis
Aggregate-only historical and fresh-holdout errors. No values, snippets, model loading, or retuning.


In [ ]:
import json
import os
from pathlib import Path

root = Path.cwd()
while not (root / "ml").is_dir() and root.parent != root:
    root = root.parent
results = Path(os.environ.get("SECRETSENSE_RESULTS_DIR", root / "ml" / "results"))
report = json.loads((results / "xgboost.json").read_text())
print("Historical errors:")
print(json.dumps(report["historical_errors"], indent=2))
print("Fresh holdout errors by group and method:")
for group, methods in report["evaluation"]["test"]["by_family"].items():
    for method, measured in methods.items():
        (tn, fp), (fn, tp) = measured["confusion_matrix"]
        print(group, method, {"TN": tn, "FP": fp, "FN": fn, "TP": tp})
print("Limitations:")
for limitation in report["limitations"]:
    print(limitation)

## Interpretation
The historical filter missed 210 database-password examples: three extraction misses and 207 additional model rejections. Rules flagged 300 authored test dummies.

On the fresh synthetic holdout both retrained models retain all 300 intended positives and reject 50 dummy-prefixed negatives, leaving 250 fixture candidates. Provider-shaped fixtures and intended credentials can share the same numeric features. These results do not establish real-world validity or justify hiding findings. XGBoost did not outperform the forest on this holdout.

The test split is now observed. Any new feature design, calibration, or threshold selection needs another reserved evaluation set and independently reviewed examples.
